# MTEX-Style Raster + ML EBSD Stitching

This notebook stitches EBSD `.ang` tiles using a map-first workflow: gridify raster EBSD maps, register IQ/CI overlap strips, score candidate shifts with FCC orientation consistency, then train a small self-supervised ML scorer to rank seam translations. Grain reconstruction should be done after this stitched map step.


## Optional Conda / Kernel Setup

Use this if the active notebook kernel does not already have the dependencies. The commands are left commented so the notebook is safe to run in an existing environment.

In [ ]:
# Optional conda setup. Run these commands in a terminal if this notebook kernel
# does not already have the dependencies installed.
#
# conda create -n ebsd-stitch python=3.11 -y
# conda activate ebsd-stitch
# python -m pip install -r requirements-ebsd-stitching.txt
# python -m ipykernel install --user --name ebsd-stitch --display-name "Python (ebsd-stitch)"
#
# In Jupyter, switch the kernel to: Python (ebsd-stitch)

import os
from pathlib import Path
cache_root = Path('.cache')
os.environ.setdefault('MPLCONFIGDIR', str(cache_root / 'matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(cache_root))
os.environ.setdefault('NUMBA_CACHE_DIR', str(cache_root / 'numba'))
print('Using local cache root:', cache_root.resolve())


## Imports and Global Constants

In [ ]:
import argparse
import json
import math
import os
import re

os.environ.setdefault('MPLCONFIGDIR', str(Path('.cache') / 'matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(Path('.cache')))
os.environ.setdefault('NUMBA_CACHE_DIR', str(Path('.cache') / 'numba'))
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

os.environ.setdefault("MPLCONFIGDIR", str(Path(".cache") / "matplotlib"))

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation
from skimage.registration import phase_cross_correlation
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


SEED = 521
RNG = np.random.default_rng(SEED)


ANG_COLUMNS = [
    "phi1",
    "Phi",
    "phi2",
    "x",
    "y",
    "IQ",
    "CI",
    "phase",
    "SEM_signal",
    "fit",
]


## Tile Data Structures

In [ ]:
@dataclass
class TileGrid:
    name: str
    row: int
    col: int
    path: Path
    x_values: np.ndarray
    y_values: np.ndarray
    x_step: float
    y_step: float
    phi1: np.ndarray
    Phi: np.ndarray
    phi2: np.ndarray
    iq: np.ndarray
    ci: np.ndarray
    phase: np.ndarray
    valid: np.ndarray

    @property
    def shape(self) -> Tuple[int, int]:
        return self.iq.shape

    @property
    def height(self) -> int:
        return self.iq.shape[0]

    @property
    def width(self) -> int:
        return self.iq.shape[1]


@dataclass
class PairCandidate:
    pair_id: str
    direction: str
    tile_a: str
    tile_b: str
    shift_y: int
    shift_x: int
    origin_dy: int
    origin_dx: int
    iq_corr: float
    ci_corr: float
    valid_fraction: float
    median_misorientation_deg: float
    p90_misorientation_deg: float
    overlap_pixels: int
    candidate_source: str
    classical_score: float
    pseudo_label: int
    ml_probability: float = float("nan")
    final_score: float = float("nan")
    selected: bool = False
    accepted: bool = False


## ANG Loading and MTEX-Style Gridify

In [ ]:
def read_ang_numeric(path: Path) -> pd.DataFrame:
    """Read numeric ANG rows into a dataframe, preserving common columns."""
    numeric_rows: List[List[float]] = []
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            parts = stripped.split()
            try:
                numeric_rows.append([float(p) for p in parts])
            except ValueError:
                continue

    if not numeric_rows:
        raise ValueError(f"No numeric ANG rows found in {path}")

    ncols = max(len(row) for row in numeric_rows)
    arr = np.full((len(numeric_rows), ncols), np.nan, dtype=float)
    for i, row in enumerate(numeric_rows):
        arr[i, : len(row)] = row

    cols = ANG_COLUMNS[:ncols] + [f"extra_{i}" for i in range(max(0, ncols - len(ANG_COLUMNS)))]
    df = pd.DataFrame(arr, columns=cols[:ncols])
    for col, default in [("IQ", 1.0), ("CI", 1.0), ("phase", 1)]:
        if col not in df:
            df[col] = default

    # ANG files are normally radians. If they look degree-like, convert.
    if df[["phi1", "Phi", "phi2"]].to_numpy(dtype=float).max() > 2 * np.pi + 1e-3:
        df[["phi1", "Phi", "phi2"]] = np.deg2rad(df[["phi1", "Phi", "phi2"]])

    df["phase"] = 1  # SCC 316 is treated as single-phase austenitic FCC.
    return df


def gridify_ang(df: pd.DataFrame, name: str, row: int, col: int, path: Path) -> TileGrid:
    """MTEX-style gridify: convert unordered EBSD points into regular arrays."""
    xs = np.sort(df["x"].dropna().unique())
    ys = np.sort(df["y"].dropna().unique())
    rectangular_fill = len(df) / max(len(xs) * len(ys), 1)
    row_counts = df.groupby("y").size()
    use_scan_rows = rectangular_fill < 0.75 and row_counts.nunique() <= 3

    if use_scan_rows:
        # Tilt-corrected or hex-like ANG exports can have staggered x positions:
        # unique x/y would create a half-empty grid. MTEX gridify preserves scan
        # rows, so use sorted points within each y row as raster columns.
        width = int(row_counts.mode().iloc[0])
        shape = (len(ys), width)
        arrays = {
            "phi1": np.full(shape, np.nan, dtype=float),
            "Phi": np.full(shape, np.nan, dtype=float),
            "phi2": np.full(shape, np.nan, dtype=float),
            "iq": np.full(shape, np.nan, dtype=float),
            "ci": np.full(shape, np.nan, dtype=float),
            "phase": np.full(shape, 1, dtype=int),
        }
        x_rows = []
        for yi, (_, gdf) in enumerate(df.sort_values(["y", "x"]).groupby("y", sort=True)):
            gdf = gdf.sort_values("x").head(width)
            n = len(gdf)
            arrays["phi1"][yi, :n] = gdf["phi1"].to_numpy()
            arrays["Phi"][yi, :n] = gdf["Phi"].to_numpy()
            arrays["phi2"][yi, :n] = gdf["phi2"].to_numpy()
            arrays["iq"][yi, :n] = gdf["IQ"].to_numpy()
            arrays["ci"][yi, :n] = gdf["CI"].to_numpy()
            arrays["phase"][yi, :n] = 1
            if n == width:
                x_rows.append(gdf["x"].to_numpy())
        x_values = np.nanmedian(np.vstack(x_rows), axis=0) if x_rows else np.arange(width, dtype=float)
    else:
        x_index = {v: i for i, v in enumerate(xs)}
        y_index = {v: i for i, v in enumerate(ys)}
        shape = (len(ys), len(xs))
        arrays = {
            "phi1": np.full(shape, np.nan, dtype=float),
            "Phi": np.full(shape, np.nan, dtype=float),
            "phi2": np.full(shape, np.nan, dtype=float),
            "iq": np.full(shape, np.nan, dtype=float),
            "ci": np.full(shape, np.nan, dtype=float),
            "phase": np.full(shape, 1, dtype=int),
        }

        for rec in df.itertuples(index=False):
            xi = x_index[getattr(rec, "x")]
            yi = y_index[getattr(rec, "y")]
            arrays["phi1"][yi, xi] = getattr(rec, "phi1")
            arrays["Phi"][yi, xi] = getattr(rec, "Phi")
            arrays["phi2"][yi, xi] = getattr(rec, "phi2")
            arrays["iq"][yi, xi] = getattr(rec, "IQ")
            arrays["ci"][yi, xi] = getattr(rec, "CI")
            arrays["phase"][yi, xi] = 1
        x_values = xs

    valid = np.isfinite(arrays["phi1"]) & np.isfinite(arrays["iq"])
    x_step = float(np.nanmedian(np.diff(x_values))) if len(x_values) > 1 else 1.0
    y_step = float(np.nanmedian(np.diff(ys))) if len(ys) > 1 else 1.0
    return TileGrid(
        name=name,
        row=row,
        col=col,
        path=path,
        x_values=x_values,
        y_values=ys,
        x_step=x_step,
        y_step=y_step,
        phi1=arrays["phi1"],
        Phi=arrays["Phi"],
        phi2=arrays["phi2"],
        iq=arrays["iq"],
        ci=arrays["ci"],
        phase=arrays["phase"],
        valid=valid,
    )


## FCC Symmetry and Orientation Math

In [ ]:
def _proper_cubic_symmetry_operators() -> np.ndarray:
    from itertools import permutations, product

    ops = []
    for perm in permutations(range(3)):
        p = np.zeros((3, 3), dtype=float)
        for row, col in enumerate(perm):
            p[row, col] = 1.0
        for signs in product([-1.0, 1.0], repeat=3):
            s = np.diag(signs) @ p
            if np.linalg.det(s) > 0.5:
                ops.append(s)
    return np.stack(ops)


CUBIC_SYMMETRY = _proper_cubic_symmetry_operators()


def euler_to_matrix(phi1: np.ndarray, Phi: np.ndarray, phi2: np.ndarray) -> np.ndarray:
    """Bunge-like Euler angles to rotation matrices, matching the existing notebook."""
    c1, s1 = np.cos(phi1), np.sin(phi1)
    c, s = np.cos(Phi), np.sin(Phi)
    c2, s2 = np.cos(phi2), np.sin(phi2)
    out = np.empty(phi1.shape + (3, 3), dtype=float)
    out[..., 0, 0] = c1 * c2 - s1 * s2 * c
    out[..., 0, 1] = s1 * c2 + c1 * s2 * c
    out[..., 0, 2] = s2 * s
    out[..., 1, 0] = -c1 * s2 - s1 * c2 * c
    out[..., 1, 1] = -s1 * s2 + c1 * c2 * c
    out[..., 1, 2] = c2 * s
    out[..., 2, 0] = s1 * s
    out[..., 2, 1] = -c1 * s
    out[..., 2, 2] = c
    return out


def cubic_misorientation_deg_many(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Approximate FCC/cubic disorientation using one-sided crystal symmetry.

    This is intentionally fast for raster scoring. It correctly collapses a
    180-degree cubic symmetry rotation to 0 degrees and is sufficient for seam
    ranking. Full two-sided disorientation can be added later if needed.
    """
    delta = np.einsum("nij,nkj->nik", a, b)
    best_trace = np.full(delta.shape[0], -3.0, dtype=float)
    for sym in CUBIC_SYMMETRY:
        sym_delta = np.einsum("ij,njk->nik", sym, delta)
        traces = np.einsum("nii->n", sym_delta)
        best_trace = np.maximum(best_trace, traces)
    cos_theta = np.clip((best_trace - 1.0) / 2.0, -1.0, 1.0)
    return np.rad2deg(np.arccos(cos_theta))


## Right/Down Overlap Registration and Candidate Scoring

In [ ]:
def normalize_image(img: np.ndarray) -> np.ndarray:
    out = img.astype(float).copy()
    finite = np.isfinite(out)
    if not finite.any():
        return np.zeros_like(out)
    med = np.nanmedian(out)
    out[~finite] = med
    lo, hi = np.nanpercentile(out, [1, 99])
    if hi <= lo:
        return np.zeros_like(out)
    out = np.clip((out - lo) / (hi - lo), 0.0, 1.0)
    return out


def pearson_corr(a: np.ndarray, b: np.ndarray, mask: np.ndarray) -> float:
    good = mask & np.isfinite(a) & np.isfinite(b)
    if good.sum() < 16:
        return 0.0
    av = a[good].astype(float)
    bv = b[good].astype(float)
    av -= av.mean()
    bv -= bv.mean()
    denom = np.linalg.norm(av) * np.linalg.norm(bv)
    return float(np.dot(av, bv) / denom) if denom > 1e-12 else 0.0


def shifted_slices(shape_a: Tuple[int, int], shape_b: Tuple[int, int], shift_y: int, shift_x: int):
    """Slices for comparing A to B after applying skimage-style shift to B."""
    ha, wa = shape_a
    hb, wb = shape_b
    ay0 = max(0, shift_y)
    ay1 = min(ha, hb + shift_y)
    ax0 = max(0, shift_x)
    ax1 = min(wa, wb + shift_x)
    if ay1 <= ay0 or ax1 <= ax0:
        return None
    by0 = ay0 - shift_y
    by1 = ay1 - shift_y
    bx0 = ax0 - shift_x
    bx1 = ax1 - shift_x
    return (slice(ay0, ay1), slice(ax0, ax1)), (slice(by0, by1), slice(bx0, bx1))


def strip_pair(a: TileGrid, b: TileGrid, direction: str, overlap_px: int):
    if direction == "horizontal":
        overlap_px = int(np.clip(overlap_px, 4, min(a.width, b.width)))
        return (
            {k: getattr(a, k)[:, -overlap_px:] for k in ["iq", "ci", "phi1", "Phi", "phi2", "valid"]},
            {k: getattr(b, k)[:, :overlap_px] for k in ["iq", "ci", "phi1", "Phi", "phi2", "valid"]},
        )
    overlap_px = int(np.clip(overlap_px, 4, min(a.height, b.height)))
    return (
        {k: getattr(a, k)[-overlap_px:, :] for k in ["iq", "ci", "phi1", "Phi", "phi2", "valid"]},
        {k: getattr(b, k)[:overlap_px, :] for k in ["iq", "ci", "phi1", "Phi", "phi2", "valid"]},
    )


def phase_corr_shift(ref: np.ndarray, moving: np.ndarray) -> Tuple[int, int]:
    ref_n = normalize_image(ref)
    mov_n = normalize_image(moving)
    shift, _, _ = phase_cross_correlation(ref_n, mov_n, upsample_factor=1, normalization=None)
    return int(round(float(shift[0]))), int(round(float(shift[1])))


def candidate_shift_set(strips_a: Dict[str, np.ndarray], strips_b: Dict[str, np.ndarray], radius: int) -> List[Tuple[int, int, str]]:
    bases = [(0, 0, "zero")]
    for channel in ["iq", "ci"]:
        try:
            sy, sx = phase_corr_shift(strips_a[channel], strips_b[channel])
            bases.append((sy, sx, f"{channel}_phase_corr"))
        except Exception:
            pass

    candidates: Dict[Tuple[int, int], str] = {}
    for sy, sx, source in bases:
        for dy in range(-radius, radius + 1):
            for dx in range(-radius, radius + 1):
                key = (sy + dy, sx + dx)
                candidates.setdefault(key, source)
    return [(sy, sx, source) for (sy, sx), source in sorted(candidates.items())]


def evaluate_candidate(
    pair_id: str,
    direction: str,
    tile_a: TileGrid,
    tile_b: TileGrid,
    strips_a: Dict[str, np.ndarray],
    strips_b: Dict[str, np.ndarray],
    shift_y: int,
    shift_x: int,
    source: str,
    overlap_px: int,
    orientation_samples: int,
) -> Optional[PairCandidate]:
    slices = shifted_slices(strips_a["iq"].shape, strips_b["iq"].shape, shift_y, shift_x)
    if slices is None:
        return None
    sa, sb = slices
    valid = strips_a["valid"][sa] & strips_b["valid"][sb]
    overlap_pixels = int(valid.sum())
    if overlap_pixels < 16:
        return None

    iq_corr = pearson_corr(strips_a["iq"][sa], strips_b["iq"][sb], valid)
    ci_corr = pearson_corr(strips_a["ci"][sa], strips_b["ci"][sb], valid)
    valid_fraction = float(overlap_pixels / valid.size)

    yy, xx = np.nonzero(valid)
    if len(yy) > orientation_samples:
        idx = RNG.choice(len(yy), size=orientation_samples, replace=False)
        yy, xx = yy[idx], xx[idx]

    a_phi1 = strips_a["phi1"][sa][yy, xx]
    a_Phi = strips_a["Phi"][sa][yy, xx]
    a_phi2 = strips_a["phi2"][sa][yy, xx]
    b_phi1 = strips_b["phi1"][sb][yy, xx]
    b_Phi = strips_b["Phi"][sb][yy, xx]
    b_phi2 = strips_b["phi2"][sb][yy, xx]
    finite_ori = np.isfinite(a_phi1) & np.isfinite(a_Phi) & np.isfinite(a_phi2) & np.isfinite(b_phi1) & np.isfinite(b_Phi) & np.isfinite(b_phi2)
    if finite_ori.sum() < 8:
        median_mis = 180.0
        p90_mis = 180.0
    else:
        ra = euler_to_matrix(a_phi1[finite_ori], a_Phi[finite_ori], a_phi2[finite_ori])
        rb = euler_to_matrix(b_phi1[finite_ori], b_Phi[finite_ori], b_phi2[finite_ori])
        mis = cubic_misorientation_deg_many(ra, rb)
        median_mis = float(np.nanmedian(mis))
        p90_mis = float(np.nanpercentile(mis, 90))

    orientation_score = math.exp(-median_mis / 8.0)
    shift_penalty = 0.015 * (abs(shift_y) + abs(shift_x))
    classical_score = (
        0.35 * max(iq_corr, 0.0)
        + 0.15 * max(ci_corr, 0.0)
        + 0.35 * orientation_score
        + 0.15 * valid_fraction
        - shift_penalty
    )

    pseudo_label = int(
        valid_fraction >= 0.60
        and iq_corr >= 0.45
        and median_mis <= 8.0
    )

    if direction == "horizontal":
        origin_dy = shift_y
        origin_dx = tile_a.width - overlap_px + shift_x
    else:
        origin_dy = tile_a.height - overlap_px + shift_y
        origin_dx = shift_x

    return PairCandidate(
        pair_id=pair_id,
        direction=direction,
        tile_a=tile_a.name,
        tile_b=tile_b.name,
        shift_y=shift_y,
        shift_x=shift_x,
        origin_dy=int(origin_dy),
        origin_dx=int(origin_dx),
        iq_corr=iq_corr,
        ci_corr=ci_corr,
        valid_fraction=valid_fraction,
        median_misorientation_deg=median_mis,
        p90_misorientation_deg=p90_mis,
        overlap_pixels=overlap_pixels,
        candidate_source=source,
        classical_score=float(classical_score),
        pseudo_label=pseudo_label,
    )


## Tile Manifest and Right/Down Pair Discovery

In [ ]:
def read_manifest(folder: Path) -> pd.DataFrame:
    manifest_path = folder / "tile_manifest.csv"
    if manifest_path.exists():
        manifest = pd.read_csv(manifest_path)
        manifest = manifest[manifest["tile_name"].astype(str).str.endswith("_clean.ang")].copy()
        if "distorted" in manifest:
            manifest = manifest[manifest["distorted"].astype(str).str.lower().isin(["false", "0", "no"])]
    else:
        rows = []
        for path in sorted(folder.glob("tile_r*_c*_clean.ang")):
            match = re.search(r"tile_r(\d+)_c(\d+)_clean\.ang$", path.name)
            if match:
                rows.append({"tile_name": path.name, "row": int(match.group(1)), "col": int(match.group(2))})
        manifest = pd.DataFrame(rows)

    if manifest.empty:
        raise ValueError(f"No clean tile manifest entries found under {folder}")

    manifest["path"] = manifest["tile_name"].map(lambda n: folder / str(n))
    manifest = manifest[manifest["path"].map(Path.exists)].copy()
    return manifest.sort_values(["row", "col"]).reset_index(drop=True)


def adjacent_pairs(manifest: pd.DataFrame) -> List[Tuple[pd.Series, pd.Series, str]]:
    by_rc = {(int(r.row), int(r.col)): r for _, r in manifest.iterrows()}
    pairs = []
    for (row, col), tile in sorted(by_rc.items()):
        right = by_rc.get((row, col + 1))
        if right is not None:
            pairs.append((tile, right, "horizontal"))
        down = by_rc.get((row + 1, col))
        if down is not None:
            pairs.append((tile, down, "vertical"))
    return pairs


def infer_overlap_px(a: TileGrid, b: TileGrid, row_a: pd.Series, row_b: pd.Series, direction: str) -> int:
    if direction == "horizontal" and {"xmin", "xmax"}.issubset(row_a.index):
        overlap_units = max(0.0, min(float(row_a.xmax), float(row_b.xmax)) - max(float(row_a.xmin), float(row_b.xmin)))
        return int(round(overlap_units / max(a.x_step, 1e-9)))
    if direction == "vertical" and {"ymin", "ymax"}.issubset(row_a.index):
        overlap_units = max(0.0, min(float(row_a.ymax), float(row_b.ymax)) - max(float(row_a.ymin), float(row_b.ymin)))
        return int(round(overlap_units / max(a.y_step, 1e-9)))
    return max(4, int(round((a.width if direction == "horizontal" else a.height) * 0.20)))


## Self-Supervised ML Seam Scorer and Placement Solver

In [ ]:
def train_ml_scorer(candidates: List[PairCandidate], use_training: bool = True, train_fraction: float = 0.70):
    """Train the self-supervised seam scorer on held-out-free pseudo-labels.

    The labels are generated from physics, not from manual annotations:
    candidates with high IQ/CI agreement and low FCC disorientation are
    pseudo-positives; poor candidates are pseudo-negatives. The model trains on
    a subset of seam pairs and predicts every candidate, including held-out
    seam pairs, so the validation report is not a same-candidate fit score.
    """
    from sklearn.metrics import average_precision_score, roc_auc_score

    feature_cols = [
        "iq_corr",
        "ci_corr",
        "valid_fraction",
        "median_misorientation_deg",
        "p90_misorientation_deg",
        "shift_abs",
        "classical_score",
    ]
    rows = []
    for c in candidates:
        rows.append(
            {
                "pair_id": c.pair_id,
                "iq_corr": c.iq_corr,
                "ci_corr": c.ci_corr,
                "valid_fraction": c.valid_fraction,
                "median_misorientation_deg": c.median_misorientation_deg,
                "p90_misorientation_deg": c.p90_misorientation_deg,
                "shift_abs": abs(c.shift_y) + abs(c.shift_x),
                "classical_score": c.classical_score,
                "pseudo_label": c.pseudo_label,
            }
        )
    df = pd.DataFrame(rows)

    def _split_metrics(split_name, mask, probs):
        y = df.loc[mask, "pseudo_label"].to_numpy(dtype=int)
        p = np.asarray(probs)[mask]
        out = {"n": int(mask.sum()), "positives": int(y.sum()), "negatives": int((1 - y).sum())}
        if len(np.unique(y)) > 1:
            out["roc_auc"] = float(roc_auc_score(y, p))
            out["pr_auc"] = float(average_precision_score(y, p))
        else:
            out["roc_auc"] = None
            out["pr_auc"] = None
        out["mean_probability"] = float(np.mean(p)) if len(p) else None
        return out

    report = {
        "mode": "self_supervised_logistic_regression" if use_training else "no_training_classical_score",
        "pseudo_label_rule": "positive when valid_fraction>=0.60, IQ_corr>=0.45, and median_FCC_misorientation<=8deg",
        "n_candidates": int(len(df)),
        "pseudo_positives": int(df["pseudo_label"].sum()) if not df.empty else 0,
        "pseudo_negatives": int((1 - df["pseudo_label"]).sum()) if not df.empty else 0,
    }

    if (not use_training) or df.empty or df["pseudo_label"].nunique() < 2 or df["pseudo_label"].sum() < 3:
        for c in candidates:
            c.ml_probability = float(np.clip(c.classical_score, 0.0, 1.0))
            c.final_score = c.ml_probability
            c.self_supervised_split = "not_trained"
        report["reason"] = "training disabled or insufficient pseudo-label diversity"
        return None, None, feature_cols, report

    pair_ids = np.array(sorted(df["pair_id"].unique()))
    rng = np.random.default_rng(SEED)
    rng.shuffle(pair_ids)
    n_train = int(np.clip(round(len(pair_ids) * train_fraction), 1, max(len(pair_ids) - 1, 1)))
    train_pairs = set(pair_ids[:n_train])
    train_mask = df["pair_id"].isin(train_pairs).to_numpy()
    val_mask = ~train_mask

    # If the pair-level split is degenerate, fall back to fitting all candidates
    # but keep the report honest.
    if df.loc[train_mask, "pseudo_label"].nunique() < 2 or df.loc[train_mask, "pseudo_label"].sum() < 3:
        train_mask = np.ones(len(df), dtype=bool)
        val_mask = np.zeros(len(df), dtype=bool)
        report["split_warning"] = "pair-level train split lacked both pseudo-label classes; fit on all candidates"

    x = df[feature_cols].to_numpy(dtype=float)
    y = df["pseudo_label"].to_numpy(dtype=int)
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x[train_mask])
    x_all = scaler.transform(x)
    model = LogisticRegression(class_weight="balanced", random_state=SEED, max_iter=1000)
    model.fit(x_train, y[train_mask])
    probs = model.predict_proba(x_all)[:, 1]

    for c, p, is_train, is_val in zip(candidates, probs, train_mask, val_mask):
        c.ml_probability = float(p)
        c.final_score = 0.65 * float(p) + 0.35 * float(np.clip(c.classical_score, 0.0, 1.0))
        c.self_supervised_split = "train" if is_train else "validation"

    report.update(
        {
            "train_fraction_by_pair": float(train_fraction),
            "n_train_pairs": int(len(set(df.loc[train_mask, "pair_id"]))),
            "n_validation_pairs": int(len(set(df.loc[val_mask, "pair_id"]))),
            "train": _split_metrics("train", train_mask, probs),
            "validation": _split_metrics("validation", val_mask, probs) if val_mask.any() else None,
        }
    )
    return model, scaler, feature_cols, report


def choose_best_candidates(
    candidates: List[PairCandidate],
    min_accept_score: float,
    max_accept_misorientation: float,
    min_valid_fraction: float,
) -> List[PairCandidate]:
    by_pair: Dict[str, List[PairCandidate]] = {}
    for c in candidates:
        by_pair.setdefault(c.pair_id, []).append(c)

    selected = []
    for pair_id, group in by_pair.items():
        best = max(group, key=lambda c: c.final_score)
        best.selected = True
        best.accepted = (
            best.final_score >= min_accept_score
            and best.median_misorientation_deg <= max_accept_misorientation
            and best.valid_fraction >= min_valid_fraction
        )
        selected.append(best)
    return selected


def solve_tile_origins(tiles: Dict[str, TileGrid], selected: List[PairCandidate]) -> Dict[str, Tuple[int, int]]:
    graph: Dict[str, List[Tuple[str, int, int]]] = {name: [] for name in tiles}
    for edge in selected:
        if not edge.accepted:
            continue
        graph[edge.tile_a].append((edge.tile_b, edge.origin_dy, edge.origin_dx))
        graph[edge.tile_b].append((edge.tile_a, -edge.origin_dy, -edge.origin_dx))

    start = min(tiles.values(), key=lambda t: (t.row, t.col)).name
    origins = {start: (0, 0)}
    queue = [start]
    while queue:
        current = queue.pop(0)
        cy, cx = origins[current]
        for nxt, dy, dx in graph[current]:
            if nxt in origins:
                continue
            origins[nxt] = (cy + dy, cx + dx)
            queue.append(nxt)

    # Disconnected fallback: place by manifest row/col pitch.
    horizontal_edges = [e.origin_dx for e in selected if e.accepted and e.direction == "horizontal"]
    vertical_edges = [e.origin_dy for e in selected if e.accepted and e.direction == "vertical"]
    widths = [t.width for t in tiles.values()]
    heights = [t.height for t in tiles.values()]
    pitch_x = int(round(np.median(horizontal_edges))) if horizontal_edges else int(round(np.median(widths) * 0.80))
    pitch_y = int(round(np.median(vertical_edges))) if vertical_edges else int(round(np.median(heights) * 0.80))
    for tile in tiles.values():
        origins.setdefault(tile.name, (tile.row * pitch_y, tile.col * pitch_x))

    min_y = min(y for y, _ in origins.values())
    min_x = min(x for _, x in origins.values())
    if min_y < 0 or min_x < 0:
        origins = {k: (y - min_y, x - min_x) for k, (y, x) in origins.items()}
    return origins


## Mosaic and Preview Writers

In [ ]:
def simple_ipf_rgb(tile: TileGrid) -> np.ndarray:
    """Fast orientation color preview; not a substitute for orix/MTEX IPF keys."""
    # Use crystal direction of sample Z in crystal frame as a stable RGB proxy.
    r = euler_to_matrix(tile.phi1, tile.Phi, tile.phi2)
    direction = np.abs(r[..., :, 2])
    direction[~tile.valid] = 0.0
    denom = np.max(direction, axis=2, keepdims=True)
    rgb = direction / np.where(denom > 1e-9, denom, 1.0)
    rgb[~tile.valid] = np.nan
    return np.clip(rgb, 0.0, 1.0)


def mosaic_tiles(tiles: Dict[str, TileGrid], origins: Dict[str, Tuple[int, int]]):
    max_y = max(origins[name][0] + tile.height for name, tile in tiles.items())
    max_x = max(origins[name][1] + tile.width for name, tile in tiles.items())
    iq = np.full((max_y, max_x), np.nan, dtype=float)
    ci = np.full((max_y, max_x), np.nan, dtype=float)
    rgb = np.full((max_y, max_x, 3), np.nan, dtype=float)
    source = np.full((max_y, max_x), "", dtype=object)

    for name, tile in sorted(tiles.items(), key=lambda kv: (kv[1].row, kv[1].col)):
        y0, x0 = origins[name]
        ys = slice(y0, y0 + tile.height)
        xs = slice(x0, x0 + tile.width)
        target_ci = ci[ys, xs]
        replace = tile.valid & (np.isnan(target_ci) | (tile.ci >= np.nan_to_num(target_ci, nan=-np.inf)))
        target_iq = iq[ys, xs]
        target_rgb = rgb[ys, xs]
        target_source = source[ys, xs]
        tile_rgb = simple_ipf_rgb(tile)
        target_iq[replace] = tile.iq[replace]
        target_ci[replace] = tile.ci[replace]
        target_rgb[replace] = tile_rgb[replace]
        target_source[replace] = name
        iq[ys, xs] = target_iq
        ci[ys, xs] = target_ci
        rgb[ys, xs] = target_rgb
        source[ys, xs] = target_source
    return iq, ci, rgb, source


def save_image(path: Path, arr: np.ndarray, cmap: str = "gray") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 8))
    if arr.ndim == 3:
        shown = np.nan_to_num(arr, nan=1.0)
        ax.imshow(shown, origin="upper")
    else:
        ax.imshow(arr, cmap=cmap, origin="upper")
    ax.set_axis_off()
    fig.tight_layout(pad=0)
    fig.savefig(path, dpi=200, bbox_inches="tight", pad_inches=0)
    plt.close(fig)


## Pipeline Runner

In [ ]:
def run(args: argparse.Namespace) -> None:
    folder = Path(args.tiles).resolve()
    out_dir = Path(args.output).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    manifest = read_manifest(folder)
    if args.max_tiles:
        manifest = manifest.head(args.max_tiles).copy()

    tiles: Dict[str, TileGrid] = {}
    for row in manifest.itertuples(index=False):
        df = read_ang_numeric(Path(row.path))
        tile = gridify_ang(df, str(row.tile_name), int(row.row), int(row.col), Path(row.path))
        tiles[tile.name] = tile

    manifest_by_name = manifest.set_index("tile_name")
    pairs = adjacent_pairs(manifest)
    if args.max_pairs:
        pairs = pairs[: args.max_pairs]

    all_candidates: List[PairCandidate] = []
    for row_a, row_b, direction in pairs:
        a = tiles[str(row_a.tile_name)]
        b = tiles[str(row_b.tile_name)]
        overlap_px = infer_overlap_px(a, b, row_a, row_b, direction)
        strips_a, strips_b = strip_pair(a, b, direction, overlap_px)
        pair_id = f"{direction}_r{int(row_a.row)}_c{int(row_a.col)}__r{int(row_b.row)}_c{int(row_b.col)}"
        for sy, sx, source in candidate_shift_set(strips_a, strips_b, args.search_radius):
            cand = evaluate_candidate(
                pair_id,
                direction,
                a,
                b,
                strips_a,
                strips_b,
                sy,
                sx,
                source,
                overlap_px,
                args.orientation_samples,
            )
            if cand is not None:
                all_candidates.append(cand)

    model, scaler, feature_cols, self_supervised_report = train_ml_scorer(
        all_candidates,
        use_training=args.use_ml_training,
        train_fraction=args.self_supervised_train_fraction,
    )
    selected = choose_best_candidates(
        all_candidates,
        min_accept_score=args.min_accept_score,
        max_accept_misorientation=args.max_accept_misorientation,
        min_valid_fraction=args.min_valid_fraction,
    )
    origins = solve_tile_origins(tiles, selected)
    iq_mosaic, ci_mosaic, rgb_mosaic, source_mosaic = mosaic_tiles(tiles, origins)

    cand_df = pd.DataFrame([c.__dict__ for c in all_candidates])
    selected_df = pd.DataFrame([c.__dict__ for c in selected])
    origins_df = pd.DataFrame(
        [{"tile_name": name, "origin_y_px": y, "origin_x_px": x, "row": tiles[name].row, "col": tiles[name].col} for name, (y, x) in origins.items()]
    ).sort_values(["row", "col"])

    cand_df.to_csv(out_dir / "candidate_shift_scores.csv", index=False)
    selected_df.to_csv(out_dir / "selected_pair_shifts.csv", index=False)
    origins_df.to_csv(out_dir / "tile_origins.csv", index=False)
    np.savez_compressed(out_dir / "stitched_mosaic_arrays.npz", iq=iq_mosaic, ci=ci_mosaic, ipf_rgb=rgb_mosaic, source=source_mosaic)
    save_image(out_dir / "stitched_iq.png", iq_mosaic, cmap="gray")
    save_image(out_dir / "stitched_ci.png", ci_mosaic, cmap="viridis")
    save_image(out_dir / "stitched_ipf_preview.png", rgb_mosaic)

    summary = {
        "tiles_folder": str(folder),
        "output_dir": str(out_dir),
        "n_tiles": len(tiles),
        "n_pairs": len(pairs),
        "n_candidates": len(all_candidates),
        "n_accepted_seams": int(sum(c.accepted for c in selected)),
        "ml_model": "logistic_regression_self_supervised" if model is not None else "classical_score_fallback",
        "feature_columns": feature_cols,
        "self_supervised_training": self_supervised_report,
        "selected_shift_summary": {
            "median_iq_corr": float(selected_df["iq_corr"].median()) if not selected_df.empty else None,
            "median_ci_corr": float(selected_df["ci_corr"].median()) if not selected_df.empty else None,
            "median_misorientation_deg": float(selected_df["median_misorientation_deg"].median()) if not selected_df.empty else None,
            "p90_misorientation_deg": float(selected_df["median_misorientation_deg"].quantile(0.90)) if not selected_df.empty else None,
            "median_valid_fraction": float(selected_df["valid_fraction"].median()) if not selected_df.empty else None,
            "accepted_seam_fraction": float(selected_df["accepted"].mean()) if not selected_df.empty else None,
        },
    }
    if model is not None:
        summary["ml_coefficients"] = {
            col: float(coef) for col, coef in zip(feature_cols, model.coef_[0])
        }
        summary["ml_intercept"] = float(model.intercept_[0])

    (out_dir / "stitch_summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))


## Where the Self-Supervision Is

No manual seam labels are used. The notebook generates pseudo-labels from EBSD physics: a candidate shift is a pseudo-positive when its overlap has enough valid pixels, high IQ agreement, and low FCC symmetry-aware disorientation. Candidate shifts that do not satisfy those checks act as pseudo-negatives. The logistic model trains on one subset of right/down seam pairs and validates on held-out seam pairs.

For another single-phase FCC material, keep the same cubic symmetry setting and point `TILES_FOLDER` to that material's clean cropped ANG tiles. If the material is multiphase or non-FCC, change the phase/symmetry section before running.


## Configure Run

In [ ]:
from types import SimpleNamespace

# SCC 316 clean cropped tiles. Change this if you want a different folder.
TILES_FOLDER = "Cropped"
OUTPUT_DIR = "raster_ml_stitch_output"

args = SimpleNamespace(
    tiles=TILES_FOLDER,
    output=OUTPUT_DIR,
    search_radius=1,
    orientation_samples=120,
    max_tiles=0,      # set >0 for quick debug
    max_pairs=0,      # set >0 for quick debug
    use_ml_training=True,
    self_supervised_train_fraction=0.70,
    min_accept_score=0.65,
    max_accept_misorientation=8.0,
    min_valid_fraction=0.60,
)
args


## Run Stitching

In [ ]:
run(args)


## Validation: Parent ANG Ground Truth

This section is self-contained. It reports pseudo-label diagnostics, but the main ground-truth result is the direct parent-ANG stitching validation: selected-shift accuracy, x/y error, true-shift rank, top-k accuracy, and score margins.


In [ ]:
"""Generate validation tables and plots for raster ML EBSD stitching outputs."""

import json
import os
from pathlib import Path
from typing import Any

try:
    from IPython.display import Image, display
except ImportError:
    def Image(filename=None, **kwargs):
        return filename
    def display(value):
        return None

os.environ.setdefault("MPLCONFIGDIR", str(Path(".cache") / "matplotlib"))

import matplotlib


matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def finite_float(value: Any) -> float | None:
    try:
        value = float(value)
    except (TypeError, ValueError):
        return None
    return value if np.isfinite(value) else None


def score_metrics(y_true: pd.Series, y_score: pd.Series, threshold: float) -> dict[str, Any]:
    y_true_arr = y_true.astype(int).to_numpy()
    y_score_arr = y_score.astype(float).to_numpy()
    y_pred = (y_score_arr >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_arr, y_pred, labels=[0, 1]).ravel()

    metrics: dict[str, Any] = {
        "threshold": threshold,
        "accuracy": finite_float(accuracy_score(y_true_arr, y_pred)),
        "precision": finite_float(precision_score(y_true_arr, y_pred, zero_division=0)),
        "recall": finite_float(recall_score(y_true_arr, y_pred, zero_division=0)),
        "f1": finite_float(f1_score(y_true_arr, y_pred, zero_division=0)),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "false_accept_rate": finite_float(fp / (fp + tn)) if (fp + tn) else None,
        "missed_accept_rate": finite_float(fn / (fn + tp)) if (fn + tp) else None,
    }
    if len(np.unique(y_true_arr)) > 1:
        metrics["roc_auc"] = finite_float(roc_auc_score(y_true_arr, y_score_arr))
        metrics["pr_auc"] = finite_float(average_precision_score(y_true_arr, y_score_arr))
    else:
        metrics["roc_auc"] = None
        metrics["pr_auc"] = None
    return metrics


def expected_calibration_error(
    y_true: pd.Series, y_score: pd.Series, n_bins: int = 10
) -> tuple[float, pd.DataFrame]:
    df = pd.DataFrame({"y_true": y_true.astype(int), "score": y_score.astype(float)})
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (df.score >= lo) & ((df.score < hi) if i < n_bins - 1 else (df.score <= hi))
        bucket = df.loc[mask]
        if bucket.empty:
            rows.append(
                {
                    "bin_left": lo,
                    "bin_right": hi,
                    "count": 0,
                    "mean_score": np.nan,
                    "empirical_rate": np.nan,
                    "abs_gap": np.nan,
                }
            )
            continue
        mean_score = float(bucket.score.mean())
        empirical_rate = float(bucket.y_true.mean())
        gap = abs(empirical_rate - mean_score)
        ece += len(bucket) / len(df) * gap
        rows.append(
            {
                "bin_left": lo,
                "bin_right": hi,
                "count": int(len(bucket)),
                "mean_score": mean_score,
                "empirical_rate": empirical_rate,
                "abs_gap": gap,
            }
        )
    return float(ece), pd.DataFrame(rows)


def add_acceptance_columns(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    df = df.copy()
    if "selected" not in df:
        df["selected"] = False
    if "accepted" not in df:
        df["accepted"] = df["final_score"].astype(float) >= threshold
    return df


def placement_residuals(selected: pd.DataFrame, origins: pd.DataFrame) -> pd.DataFrame:
    origin_by_tile = origins.set_index("tile_name")[["origin_y_px", "origin_x_px"]]
    rows = []
    for row in selected.itertuples(index=False):
        if not bool(getattr(row, "accepted", False)):
            continue
        if row.tile_a not in origin_by_tile.index or row.tile_b not in origin_by_tile.index:
            continue
        a = origin_by_tile.loc[row.tile_a]
        b = origin_by_tile.loc[row.tile_b]
        rows.append(
            {
                "pair_id": row.pair_id,
                "direction": row.direction,
                "residual_y_px": float((b.origin_y_px - a.origin_y_px) - row.origin_dy),
                "residual_x_px": float((b.origin_x_px - a.origin_x_px) - row.origin_dx),
            }
        )
    return pd.DataFrame(rows)


def infer_ang_grid_steps(path: Path) -> tuple[float, float]:
    rows: list[tuple[float, float]] = []
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            parts = stripped.split()
            if len(parts) < 5:
                continue
            try:
                rows.append((float(parts[3]), float(parts[4])))
            except ValueError:
                continue

    if not rows:
        return 1.0, 1.0
    xy = pd.DataFrame(rows, columns=["x", "y"])
    x_unique = np.sort(xy["x"].unique())
    y_unique = np.sort(xy["y"].unique())
    rectangular_fill = len(xy) / max(len(x_unique) * len(y_unique), 1)
    row_counts = xy.groupby("y").size()
    use_scan_rows = rectangular_fill < 0.75 and row_counts.nunique() <= 3

    if use_scan_rows:
        width = int(row_counts.mode().iloc[0])
        x_rows = []
        for _, group in xy.sort_values(["y", "x"]).groupby("y", sort=True):
            row_x = group.sort_values("x")["x"].to_numpy(dtype=float)[:width]
            if len(row_x) == width:
                x_rows.append(row_x)
        x_values = np.nanmedian(np.vstack(x_rows), axis=0) if x_rows else x_unique
    else:
        x_values = x_unique

    x_unique = np.sort(np.unique(np.asarray(x_values, dtype=float)))
    x_diffs = np.diff(x_unique)
    y_diffs = np.diff(y_unique)
    x_diffs = x_diffs[x_diffs > 1e-9]
    y_diffs = y_diffs[y_diffs > 1e-9]
    x_step = float(np.nanmedian(x_diffs)) if len(x_diffs) else 1.0
    y_step = float(np.nanmedian(y_diffs)) if len(y_diffs) else 1.0
    return x_step, y_step


def read_clean_manifest(tiles_folder: Path) -> pd.DataFrame:
    manifest_path = tiles_folder / "tile_manifest.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Missing tile manifest: {manifest_path}")
    manifest = pd.read_csv(manifest_path)
    manifest = manifest[manifest["tile_name"].astype(str).str.endswith("_clean.ang")].copy()
    if "distorted" in manifest:
        manifest = manifest[
            manifest["distorted"].astype(str).str.lower().isin(["false", "0", "no"])
        ].copy()
    if manifest.empty:
        raise ValueError(f"No clean tile entries found in {manifest_path}")
    return manifest


def tile_min_xy(path: Path) -> tuple[float, float]:
    xs: list[float] = []
    ys: list[float] = []
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            parts = stripped.split()
            if len(parts) < 5:
                continue
            try:
                xs.append(float(parts[3]))
                ys.append(float(parts[4]))
            except ValueError:
                continue
    if not xs or not ys:
        raise ValueError(f"No numeric x/y coordinates found in {path}")
    return min(xs), min(ys)


def parent_manifest_ground_truth(
    candidates: pd.DataFrame, tiles_folder: Path, tolerance_px: int
) -> tuple[pd.DataFrame, dict[str, Any]]:
    manifest = read_clean_manifest(tiles_folder)
    first_tile = tiles_folder / str(manifest.iloc[0]["tile_name"])
    x_step, y_step = infer_ang_grid_steps(first_tile)
    min_x = float(manifest["xmin"].min()) if "xmin" in manifest else 0.0
    min_y = float(manifest["ymin"].min()) if "ymin" in manifest else 0.0

    manifest["ground_truth_origin_x_px"] = (
        (manifest["xmin"].astype(float) - min_x) / max(x_step, 1e-9)
    ).round().astype(int)
    manifest["ground_truth_origin_y_px"] = (
        (manifest["ymin"].astype(float) - min_y) / max(y_step, 1e-9)
    ).round().astype(int)

    origin_lookup = manifest.set_index("tile_name")[
        ["ground_truth_origin_y_px", "ground_truth_origin_x_px"]
    ]
    rows = []
    for row in candidates.itertuples(index=False):
        if row.tile_a not in origin_lookup.index or row.tile_b not in origin_lookup.index:
            gt_dy = np.nan
            gt_dx = np.nan
            err_y = np.nan
            err_x = np.nan
            label = 0
        else:
            a = origin_lookup.loc[row.tile_a]
            b = origin_lookup.loc[row.tile_b]
            gt_dy = int(b.ground_truth_origin_y_px - a.ground_truth_origin_y_px)
            gt_dx = int(b.ground_truth_origin_x_px - a.ground_truth_origin_x_px)
            err_y = int(row.origin_dy - gt_dy)
            err_x = int(row.origin_dx - gt_dx)
            label = int(abs(err_y) <= tolerance_px and abs(err_x) <= tolerance_px)
        rows.append(
            {
                "ground_truth_origin_dy": gt_dy,
                "ground_truth_origin_dx": gt_dx,
                "ground_truth_error_y_px": err_y,
                "ground_truth_error_x_px": err_x,
                "ground_truth_label": label,
            }
        )

    labelled = pd.concat([candidates.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    summary = {
        "tiles_folder": str(tiles_folder),
        "source": "tile_manifest.csv parent crop coordinates",
        "tolerance_px": int(tolerance_px),
        "x_step": finite_float(x_step),
        "y_step": finite_float(y_step),
        "n_candidates_with_ground_truth": int(len(labelled)),
        "ground_truth_positives": int(labelled["ground_truth_label"].sum()),
        "ground_truth_negatives": int((1 - labelled["ground_truth_label"]).sum()),
    }
    return labelled, summary


def parent_ang_ground_truth(
    candidates: pd.DataFrame,
    tiles_folder: Path,
    parent_ang: Path,
    tolerance_px: int,
    tilt_degrees: float | None,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    parent_x_step, parent_y_step = infer_ang_grid_steps(parent_ang)
    tilt_factor = None
    if tilt_degrees is not None:
        tilt_factor = 1.0 / np.cos(np.deg2rad(float(tilt_degrees)))
        parent_y_step *= tilt_factor

    tile_origin_cache: dict[str, tuple[float, float]] = {}

    def origin_for_tile(tile_name: str) -> tuple[float, float]:
        if tile_name not in tile_origin_cache:
            tile_origin_cache[tile_name] = tile_min_xy(tiles_folder / tile_name)
        return tile_origin_cache[tile_name]

    rows = []
    for row in candidates.itertuples(index=False):
        try:
            ax, ay = origin_for_tile(row.tile_a)
            bx, by = origin_for_tile(row.tile_b)
            gt_dx = int(round((bx - ax) / max(parent_x_step, 1e-9)))
            gt_dy = int(round((by - ay) / max(parent_y_step, 1e-9)))
            err_y = int(row.origin_dy - gt_dy)
            err_x = int(row.origin_dx - gt_dx)
            label = int(abs(err_y) <= tolerance_px and abs(err_x) <= tolerance_px)
        except Exception:
            gt_dy = np.nan
            gt_dx = np.nan
            err_y = np.nan
            err_x = np.nan
            label = 0
        rows.append(
            {
                "parent_ang_origin_dy": gt_dy,
                "parent_ang_origin_dx": gt_dx,
                "parent_ang_error_y_px": err_y,
                "parent_ang_error_x_px": err_x,
                "parent_ang_label": label,
            }
        )

    labelled = pd.concat([candidates.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    summary = {
        "parent_ang": str(parent_ang),
        "tiles_folder": str(tiles_folder),
        "source": "direct tile coordinates compared against parent ANG grid spacing",
        "tolerance_px": int(tolerance_px),
        "parent_x_step": finite_float(parent_x_step),
        "parent_y_step_after_tilt": finite_float(parent_y_step),
        "tilt_degrees": finite_float(tilt_degrees) if tilt_degrees is not None else None,
        "tilt_factor": finite_float(tilt_factor),
        "n_candidates_with_ground_truth": int(len(labelled)),
        "ground_truth_positives": int(labelled["parent_ang_label"].sum()),
        "ground_truth_negatives": int((1 - labelled["parent_ang_label"]).sum()),
    }
    return labelled, summary


def layout_validation_from_origins(origins: pd.DataFrame) -> dict[str, Any] | None:
    required = {"row", "col", "origin_y_px", "origin_x_px"}
    if not required.issubset(origins.columns):
        return None

    x_steps = origins.sort_values(["row", "col"]).groupby("row").origin_x_px.diff()
    y_steps = origins.sort_values(["col", "row"]).groupby("col").origin_y_px.diff()
    x_step = float(x_steps[x_steps > 0].median()) if (x_steps > 0).any() else 0.0
    y_step = float(y_steps[y_steps > 0].median()) if (y_steps > 0).any() else 0.0
    expected_x = (origins["col"] - origins["col"].min()) * x_step
    expected_y = (origins["row"] - origins["row"].min()) * y_step
    dx = origins["origin_x_px"] - expected_x
    dy = origins["origin_y_px"] - expected_y

    return {
        "n_tiles_compared": int(len(origins)),
        "x_step_px": finite_float(x_step),
        "y_step_px": finite_float(y_step),
        "mean_abs_origin_dx_error_px": finite_float(dx.abs().mean()),
        "mean_abs_origin_dy_error_px": finite_float(dy.abs().mean()),
        "max_abs_origin_dx_error_px": finite_float(dx.abs().max()),
        "max_abs_origin_dy_error_px": finite_float(dy.abs().max()),
    }


def save_score_confusion_plot(
    path: Path,
    y_true: pd.Series,
    y_score: pd.Series,
    threshold: float,
    title: str,
    ylabel: str,
) -> None:
    y_pred = (y_score.astype(float).to_numpy() >= threshold).astype(int)
    cm = confusion_matrix(y_true.astype(int).to_numpy(), y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    for (i, j), value in np.ndenumerate(cm):
        ax.text(j, i, str(int(value)), ha="center", va="center", color="black")
    ax.set_xticks([0, 1], ["reject", "accept"])
    ax.set_yticks([0, 1], ["reject", "accept"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def save_confusion_plot(path: Path, y_true: pd.Series, y_score: pd.Series, threshold: float) -> None:
    save_score_confusion_plot(
        path,
        y_true,
        y_score,
        threshold,
        "Final score confusion matrix",
        "Pseudo label",
    )


def save_validation_plots(
    validation_dir: Path,
    candidates: pd.DataFrame,
    selected: pd.DataFrame,
    calibration_table: pd.DataFrame,
    ece: float,
    residual_df: pd.DataFrame,
    threshold: float,
) -> None:
    save_confusion_plot(
        validation_dir / "confusion_matrix_final_score.png",
        candidates["pseudo_label"],
        candidates["final_score"],
        threshold,
    )

    valid_bins = calibration_table[calibration_table["count"] > 0]
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(valid_bins.mean_score, valid_bins.empirical_rate, "o-", label=f"ECE={ece:.3f}")
    ax.plot([0, 1], [0, 1], "k--", linewidth=1)
    ax.set_xlabel("Mean score")
    ax.set_ylabel("Empirical pseudo-positive rate")
    ax.set_title("Self-supervised seam score calibration")
    ax.legend()
    fig.savefig(validation_dir / "calibration_curve_final_score.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

    direction_summary = (
        selected.groupby("direction")
        .agg(n_pairs=("pair_id", "count"), accepted=("accepted", "sum"))
        .reset_index()
    )
    direction_summary["acceptance_rate"] = direction_summary["accepted"] / direction_summary["n_pairs"]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(direction_summary.direction, direction_summary.acceptance_rate)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Accepted selected seams")
    ax.set_title("Seam acceptance by direction")
    fig.savefig(validation_dir / "seam_acceptance_by_direction.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    selected.boxplot(column="median_misorientation_deg", by="direction", ax=ax)
    ax.set_title("Selected seam median misorientation")
    ax.set_xlabel("Direction")
    ax.set_ylabel("Degrees")
    fig.suptitle("")
    fig.savefig(validation_dir / "misorientation_by_direction.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

    if not residual_df.empty:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.scatter(residual_df.residual_x_px, residual_df.residual_y_px, alpha=0.8)
        ax.axhline(0, color="k", linewidth=0.8)
        ax.axvline(0, color="k", linewidth=0.8)
        ax.set_xlabel("Residual x (px)")
        ax.set_ylabel("Residual y (px)")
        ax.set_title("Accepted seam placement residuals")
        fig.savefig(validation_dir / "placement_residuals.png", dpi=180, bbox_inches="tight")
        plt.close(fig)


def validate_output(
    output_dir: Path,
    threshold: float,
    tiles_folder: Path | None = None,
    ground_truth_tolerance_px: int = 0,
    parent_ang: Path | None = None,
    parent_tilt_degrees: float | None = 70.0,
) -> dict[str, Any]:
    validation_dir = output_dir / "validation"
    validation_dir.mkdir(parents=True, exist_ok=True)

    candidates = add_acceptance_columns(
        pd.read_csv(output_dir / "candidate_shift_scores.csv"), threshold
    )
    selected = add_acceptance_columns(pd.read_csv(output_dir / "selected_pair_shifts.csv"), threshold)
    origins = pd.read_csv(output_dir / "tile_origins.csv")

    direction_summary = (
        selected.groupby("direction")
        .agg(
            n_pairs=("pair_id", "count"),
            accepted=("accepted", "sum"),
            median_misorientation_deg=("median_misorientation_deg", "median"),
            p90_misorientation_deg=("p90_misorientation_deg", "median"),
            median_iq_corr=("iq_corr", "median"),
            median_ci_corr=("ci_corr", "median"),
            median_final_score=("final_score", "median"),
        )
        .reset_index()
    )
    direction_summary["acceptance_rate"] = (
        direction_summary["accepted"] / direction_summary["n_pairs"]
    )
    direction_summary.to_csv(validation_dir / "direction_summary.csv", index=False)

    metric_columns = [
        col for col in ["classical_score", "ml_probability", "final_score"] if col in candidates
    ]
    metrics = {
        col: score_metrics(candidates["pseudo_label"], candidates[col], threshold)
        for col in metric_columns
    }

    ece, calibration_table = expected_calibration_error(
        candidates["pseudo_label"], candidates["final_score"], n_bins=10
    )
    calibration_table.to_csv(validation_dir / "calibration_table.csv", index=False)

    residual_df = placement_residuals(selected, origins)
    residual_df.to_csv(validation_dir / "placement_residuals.csv", index=False)
    save_validation_plots(
        validation_dir, candidates, selected, calibration_table, ece, residual_df, threshold
    )

    placement_summary = {
        "n_edges": int(len(residual_df)),
        "mean_abs_x_px": finite_float(residual_df.residual_x_px.abs().mean())
        if not residual_df.empty
        else None,
        "mean_abs_y_px": finite_float(residual_df.residual_y_px.abs().mean())
        if not residual_df.empty
        else None,
        "max_abs_x_px": finite_float(residual_df.residual_x_px.abs().max())
        if not residual_df.empty
        else None,
        "max_abs_y_px": finite_float(residual_df.residual_y_px.abs().max())
        if not residual_df.empty
        else None,
    }

    ground_truth_summary = None
    if tiles_folder is not None:
        gt_candidates, ground_truth_summary = parent_manifest_ground_truth(
            candidates, tiles_folder, ground_truth_tolerance_px
        )
        gt_candidates.to_csv(validation_dir / "ground_truth_candidate_labels.csv", index=False)
        ground_truth_summary["score_metrics"] = {
            col: score_metrics(gt_candidates["ground_truth_label"], gt_candidates[col], threshold)
            for col in metric_columns
        }
        save_score_confusion_plot(
            validation_dir / "confusion_matrix_final_score_ground_truth.png",
            gt_candidates["ground_truth_label"],
            gt_candidates["final_score"],
            threshold,
            "Final score vs parent-manifest ground truth",
            "Ground truth",
        )

        selected_gt = gt_candidates[gt_candidates["selected"].astype(bool)].copy()
        selected_gt.to_csv(validation_dir / "ground_truth_selected_pair_shifts.csv", index=False)
        if not selected_gt.empty:
            accepted = selected_gt["accepted"].astype(bool)
            gt_label = selected_gt["ground_truth_label"].astype(bool)
            ground_truth_summary["selected_pairs"] = {
                "n_selected": int(len(selected_gt)),
                "ground_truth_correct_selected": int(gt_label.sum()),
                "ground_truth_correct_selected_fraction": finite_float(gt_label.mean()),
                "accepted_ground_truth_correct": int((accepted & gt_label).sum()),
                "rejected_ground_truth_correct": int((~accepted & gt_label).sum()),
            }

    parent_ang_summary = None
    if tiles_folder is not None and parent_ang is not None:
        parent_candidates, parent_ang_summary = parent_ang_ground_truth(
            candidates,
            tiles_folder,
            parent_ang,
            ground_truth_tolerance_px,
            parent_tilt_degrees,
        )
        parent_candidates.to_csv(validation_dir / "parent_ang_candidate_labels.csv", index=False)
        parent_ang_summary["score_metrics"] = {
            col: score_metrics(parent_candidates["parent_ang_label"], parent_candidates[col], threshold)
            for col in metric_columns
        }
        save_score_confusion_plot(
            validation_dir / "confusion_matrix_final_score_parent_ang.png",
            parent_candidates["parent_ang_label"],
            parent_candidates["final_score"],
            threshold,
            "Final score vs parent ANG ground truth",
            "Parent ANG truth",
        )

        selected_parent = parent_candidates[parent_candidates["selected"].astype(bool)].copy()
        selected_parent.to_csv(validation_dir / "parent_ang_selected_pair_shifts.csv", index=False)
        if not selected_parent.empty:
            accepted = selected_parent["accepted"].astype(bool)
            parent_label = selected_parent["parent_ang_label"].astype(bool)
            parent_ang_summary["selected_pairs"] = {
                "n_selected": int(len(selected_parent)),
                "parent_ang_correct_selected": int(parent_label.sum()),
                "parent_ang_correct_selected_fraction": finite_float(parent_label.mean()),
                "accepted_parent_ang_correct": int((accepted & parent_label).sum()),
                "rejected_parent_ang_correct": int((~accepted & parent_label).sum()),
            }

    summary = {
        "n_candidates": int(len(candidates)),
        "n_selected_pairs": int(len(selected)),
        "n_accepted_seams": int(selected["accepted"].sum()),
        "accepted_seam_fraction": finite_float(selected["accepted"].mean()),
        "direction_summary": direction_summary.to_dict(orient="records"),
        "score_metrics": metrics,
        "calibration": {"ECE": finite_float(ece)},
        "placement_residuals": placement_summary,
        "origin_grid_validation": layout_validation_from_origins(origins),
        "parent_manifest_ground_truth": ground_truth_summary,
        "parent_ang_ground_truth": parent_ang_summary,
        "note": (
            "Pseudo-label metrics are self-supervised seam diagnostics, not external "
            "ground truth. For older outputs without an accepted column, accepted is "
            f"derived as final_score >= {threshold}."
        ),
    }
    (validation_dir / "validation_summary.json").write_text(json.dumps(summary, indent=2))
    return summary


def summarize_parent_ang_stitching_validation(
    validation_dir: Path,
    score_column: str = "final_score",
    top_k_values: tuple[int, ...] = (1, 2, 3, 5),
) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame]:
    """Evaluate parent-ANG ground truth as a stitching/ranking task."""
    candidates = pd.read_csv(validation_dir / "parent_ang_candidate_labels.csv")
    selected = pd.read_csv(validation_dir / "parent_ang_selected_pair_shifts.csv")

    selected["abs_parent_ang_error_y_px"] = selected["parent_ang_error_y_px"].abs()
    selected["abs_parent_ang_error_x_px"] = selected["parent_ang_error_x_px"].abs()
    selected["parent_ang_linf_error_px"] = selected[
        ["abs_parent_ang_error_y_px", "abs_parent_ang_error_x_px"]
    ].max(axis=1)

    pair_rows = []
    for pair_id, group in candidates.groupby("pair_id", sort=False):
        group = group.copy().sort_values(score_column, ascending=False).reset_index(drop=True)
        group["rank"] = np.arange(1, len(group) + 1)
        true_rows = group[group["parent_ang_label"].astype(int) == 1]
        selected_rows = group[group["selected"].astype(bool)]
        best = group.iloc[0]
        selected_row = selected_rows.iloc[0] if not selected_rows.empty else None
        true_row = true_rows.sort_values(score_column, ascending=False).iloc[0] if not true_rows.empty else None
        best_false = group[group["parent_ang_label"].astype(int) == 0].head(1)
        best_false_score = float(best_false[score_column].iloc[0]) if not best_false.empty else np.nan

        if true_row is None:
            true_rank = np.nan
            true_score = np.nan
            true_error_y = np.nan
            true_error_x = np.nan
        else:
            true_rank = int(true_row["rank"])
            true_score = float(true_row[score_column])
            true_error_y = int(true_row["parent_ang_error_y_px"])
            true_error_x = int(true_row["parent_ang_error_x_px"])

        if selected_row is None:
            selected_is_true = False
            selected_score = np.nan
            selected_error_y = np.nan
            selected_error_x = np.nan
            selected_accepted = False
        else:
            selected_is_true = bool(int(selected_row["parent_ang_label"]) == 1)
            selected_score = float(selected_row[score_column])
            selected_error_y = int(selected_row["parent_ang_error_y_px"])
            selected_error_x = int(selected_row["parent_ang_error_x_px"])
            selected_accepted = bool(selected_row.get("accepted", False))

        pair_rows.append(
            {
                "pair_id": pair_id,
                "direction": str(group.iloc[0]["direction"]),
                "n_candidates": int(len(group)),
                "true_rank": true_rank,
                "true_score": true_score,
                "best_score": float(best[score_column]),
                "best_is_true": bool(int(best["parent_ang_label"]) == 1),
                "selected_is_true": selected_is_true,
                "selected_accepted": selected_accepted,
                "selected_score": selected_score,
                "selected_error_y_px": selected_error_y,
                "selected_error_x_px": selected_error_x,
                "true_error_y_px": true_error_y,
                "true_error_x_px": true_error_x,
                "best_false_score": best_false_score,
                "true_minus_best_false_margin": finite_float(true_score - best_false_score),
                "selected_minus_best_false_margin": finite_float(selected_score - best_false_score),
            }
        )

    pair_df = pd.DataFrame(pair_rows)
    pair_df.to_csv(validation_dir / "parent_ang_pair_ranking_validation.csv", index=False)

    direction_summary = (
        selected.groupby("direction")
        .agg(
            n_pairs=("pair_id", "count"),
            selected_correct=("parent_ang_label", "sum"),
            mean_abs_y_error_px=("abs_parent_ang_error_y_px", "mean"),
            mean_abs_x_error_px=("abs_parent_ang_error_x_px", "mean"),
            max_linf_error_px=("parent_ang_linf_error_px", "max"),
            accepted=("accepted", "sum"),
        )
        .reset_index()
    )
    direction_summary["selected_accuracy"] = (
        direction_summary["selected_correct"] / direction_summary["n_pairs"]
    )
    direction_summary.to_csv(validation_dir / "parent_ang_selected_accuracy_by_direction.csv", index=False)

    valid_ranks = pair_df["true_rank"].dropna().astype(int)
    summary = {
        "metric_type": "parent ANG top-1 selected shift and ranking validation",
        "score_column": score_column,
        "n_pairs": int(len(pair_df)),
        "selected_top1_accuracy": finite_float(pair_df["selected_is_true"].mean()),
        "best_score_top1_accuracy": finite_float(pair_df["best_is_true"].mean()),
        "mean_abs_selected_y_error_px": finite_float(pair_df["selected_error_y_px"].abs().mean()),
        "mean_abs_selected_x_error_px": finite_float(pair_df["selected_error_x_px"].abs().mean()),
        "max_abs_selected_y_error_px": finite_float(pair_df["selected_error_y_px"].abs().max()),
        "max_abs_selected_x_error_px": finite_float(pair_df["selected_error_x_px"].abs().max()),
        "mean_true_rank": finite_float(valid_ranks.mean()),
        "median_true_rank": finite_float(valid_ranks.median()),
        "mean_reciprocal_rank": finite_float((1.0 / valid_ranks).mean()),
        "mean_true_minus_best_false_margin": finite_float(pair_df["true_minus_best_false_margin"].mean()),
        "min_true_minus_best_false_margin": finite_float(pair_df["true_minus_best_false_margin"].min()),
        "accepted_parent_correct": int((pair_df["selected_accepted"] & pair_df["selected_is_true"]).sum()),
        "rejected_parent_correct": int(((~pair_df["selected_accepted"]) & pair_df["selected_is_true"]).sum()),
        "direction_summary": direction_summary.to_dict(orient="records"),
        "note": "Use these metrics as the main parent-ground-truth validation. Candidate threshold confusion is secondary and can overcount near-shift false positives.",
    }
    for k in top_k_values:
        summary[f"top{k}_true_shift_accuracy"] = finite_float((valid_ranks <= k).mean())

    (validation_dir / "parent_ang_ranking_summary.json").write_text(json.dumps(summary, indent=2))
    return summary, pair_df, selected


def save_parent_ang_ground_truth_plots(
    validation_dir: Path,
    ranking_df: pd.DataFrame,
    selected_df: pd.DataFrame,
) -> None:
    selected_df = selected_df.copy()
    selected_df["abs_parent_ang_error_y_px"] = selected_df["parent_ang_error_y_px"].abs()
    selected_df["abs_parent_ang_error_x_px"] = selected_df["parent_ang_error_x_px"].abs()

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(
        selected_df["parent_ang_error_x_px"],
        selected_df["parent_ang_error_y_px"],
        c=selected_df["accepted"].astype(int),
        cmap="coolwarm",
        alpha=0.85,
    )
    ax.axhline(0, color="k", linewidth=0.8)
    ax.axvline(0, color="k", linewidth=0.8)
    ax.set_xlabel("Selected x error vs parent ANG (px)")
    ax.set_ylabel("Selected y error vs parent ANG (px)")
    ax.set_title("Selected shift error vs parent ANG")
    fig.savefig(validation_dir / "parent_ang_selected_shift_error.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4))
    bins = np.arange(0.5, ranking_df["true_rank"].max() + 1.5, 1)
    ax.hist(ranking_df["true_rank"].dropna(), bins=bins, edgecolor="black")
    ax.set_xlabel("Rank of true parent shift by final score")
    ax.set_ylabel("Number of seams")
    ax.set_title("True-shift rank distribution")
    fig.savefig(validation_dir / "parent_ang_true_rank_histogram.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    ranking_df.boxplot(column="true_minus_best_false_margin", by="direction", ax=ax)
    ax.axhline(0, color="k", linewidth=0.8)
    ax.set_xlabel("Direction")
    ax.set_ylabel("True score - best false score")
    ax.set_title("Parent-true score margin by direction")
    fig.suptitle("")
    fig.savefig(validation_dir / "parent_ang_score_margin_by_direction.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


# Parent ANG ground-truth validation for the SCC 316 cropped tiles.
PARENT_ANG = Path("EBSD stiching dev dataset/5%coldrolled_30min_975C_65mag2.ang")
PARENT_TILT_DEGREES = 70.0
GROUND_TRUTH_TOLERANCE_PX = 0

validation_summary = validate_output(
    Path(OUTPUT_DIR),
    threshold=args.min_accept_score,
    tiles_folder=Path(TILES_FOLDER),
    ground_truth_tolerance_px=GROUND_TRUTH_TOLERANCE_PX,
    parent_ang=PARENT_ANG,
    parent_tilt_degrees=PARENT_TILT_DEGREES,
)

validation_dir = Path(OUTPUT_DIR) / "validation"
ranking_summary, parent_ranking_df, parent_selected_df = summarize_parent_ang_stitching_validation(
    validation_dir,
    score_column="final_score",
)
save_parent_ang_ground_truth_plots(validation_dir, parent_ranking_df, parent_selected_df)
validation_summary["parent_ang_ranking_validation"] = ranking_summary
(validation_dir / "validation_summary.json").write_text(json.dumps(validation_summary, indent=2))

print("Parent ANG stitching ground-truth summary")
print(json.dumps(ranking_summary, indent=2))
print("\nFull validation summary")
print(json.dumps(validation_summary, indent=2))

parent_misses = parent_selected_df[parent_selected_df["parent_ang_label"] == 0]
print(f"Selected seams parent-ANG misses: {len(parent_misses)} / {len(parent_selected_df)}")
display(parent_misses[[
    "pair_id", "direction", "accepted", "origin_dy", "origin_dx",
    "parent_ang_origin_dy", "parent_ang_origin_dx",
    "parent_ang_error_y_px", "parent_ang_error_x_px",
    "median_misorientation_deg", "final_score"
]].head(40))

print("Lowest parent-true score margins")
display(parent_ranking_df.sort_values("true_minus_best_false_margin").head(20))

for image_name in [
    "parent_ang_selected_shift_error.png",
    "parent_ang_true_rank_histogram.png",
    "parent_ang_score_margin_by_direction.png",
    "confusion_matrix_final_score_parent_ang.png",
    "confusion_matrix_final_score.png",
    "calibration_curve_final_score.png",
    "seam_acceptance_by_direction.png",
    "placement_residuals.png",
]:
    image_path = validation_dir / image_name
    if image_path.exists():
        print(image_name)
        display(Image(filename=str(image_path)))

## Inspect Results

In [ ]:
from IPython.display import Image, display
import json
from pathlib import Path
import pandas as pd

out = Path(OUTPUT_DIR)
summary = json.loads((out / "stitch_summary.json").read_text())
print(json.dumps(summary, indent=2))

selected = pd.read_csv(out / "selected_pair_shifts.csv")
display(selected[[
    "pair_id", "direction", "accepted", "shift_y", "shift_x",
    "origin_dy", "origin_dx", "iq_corr", "ci_corr",
    "median_misorientation_deg", "ml_probability", "final_score"
]].sort_values(["accepted", "median_misorientation_deg"], ascending=[True, False]).head(25))

for image_name in ["stitched_iq.png", "stitched_ci.png", "stitched_ipf_preview.png"]:
    print(image_name)
    display(Image(filename=str(out / image_name)))
